# Re-run: thinning the training data (RQ4)

For each model and corpus, Table 7 is the mean test AUC of a regime's fits (five folds; at 10% and 2%, five
draws of five folds) minus the mean of the five fits with all training learners, computed before rounding.
This notebook recomputes the table and the ranges the text states. It needs only the standard library.

In the per-case outputs, the model the article calls Best-LR with time windows has the identifier `Best-LR+`.

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os, sys
from pathlib import Path

# Importing from the release's code/ folder must not leave __pycache__ folders in it.
sys.dont_write_bytecode = True

RELEASE_ID = "knowledge-tracing-benchmark-2026"   # set when the report is built
DATA_NAME = 'per-case-outputs.jsonl'     # this release's per-case file, set when the report is built (None: it has none)


def find_release_file(name=None):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    name = name or DATA_NAME
    if not name:
        raise FileNotFoundError(f"Release {RELEASE_ID} publishes no per-case file; this notebook reads its "
                                f"code package (code_dir).")
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        if (d / name).is_file():
            return d / name
        gz = d / (name + ".gz")
        if gz.is_file():
            # The report offers the data file gzipped; unpack it once beside the download, so a notebook that
            # opens the file as text reads the same bytes either way.
            try:
                (d / name).write_bytes(gzip.decompress(gz.read_bytes()))
                return d / name
            except OSError:
                return gz
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def is_this_release(d):
    """True when folder d holds this release, not another one's code/ found higher up: d is named after the
    release, or its code-manifest.txt or a release file (*.yaml) names it."""
    if d.name == RELEASE_ID:
        return True
    texts = [d / "code-manifest.txt", *sorted(d.glob("*.yaml"))]
    return any(f.is_file() and RELEASE_ID in f.read_text(encoding="utf-8", errors="ignore") for f in texts)


def code_dir():
    """The release's code package (its code/ folder), looked for only by a notebook that reads it: in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file() and is_this_release(d):
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


print("Data:", shown(find_release_file()))

Data: per-case-outputs.jsonl


In [2]:
from statistics import fmean
recs = [json.loads(x) for x in open(find_release_file("per-case-outputs.jsonl"), encoding="utf-8")]
fits = [r for r in recs if r["record"] == "fit"]
C = {"half A": "kddcup2010-algebra0607:halfA", "bridge06": "kddcup2010-bridge06", "EDM Cup": "edmcup2023"}
by = {}
for r in fits:
    by.setdefault((r["model"], r["corpus"], r["regime"]), []).append(r["auc"])
loss = lambda m, c, g: fmean(by[(m, C[c], "main")]) - fmean(by[(m, C[c], g)])
for (m, cc, g), v in by.items():
    assert len(v) == {"main": 5, "seed1": 1, "quarter": 5, "tenth": 25, "fiftieth": 25,
                      "history-fifty": 5, "history-twenty": 5}[g], (m, cc, g, len(v))
s3 = lambda x: f"{-x:+.3f}".replace("-", "−")
assert [s3(loss("AKT", "half A", g)) for g in ("quarter", "tenth", "fiftieth", "history-fifty", "history-twenty")] == \
    ["−0.026", "−0.050", "−0.093", "−0.091", "−0.110"]
assert s3(loss("LPKT", "half A", "quarter")) == "−0.168"
assert all(abs(loss("running-proportion", c, g)) < 1e-12 for c in C
           for g in ("quarter", "tenth", "fiftieth", "history-fifty", "history-twenty"))
print("Table 7's checked rows match; the running proportion does not change.")

Table 7's checked rows match; the running proportion does not change.


In [3]:
def rng(cells):
    v = [loss(*x) for x in cells]
    return f"{min(v):.3f}", f"{max(v):.3f}"
DEEP = ["DKT", "AKT", "simpleKT", "IEKT", "HawkesKT", "LPKT"]
LOG = ["PFA", "Best-LR", "Best-LR-iter1000", "Best-LR+"]
present = lambda m, c: (m, C[c], "main") in by
deep2 = [(m, c, "fiftieth") for m in DEEP for c in C if present(m, c) and not (m == "HawkesKT" and c != "EDM Cup")]
assert rng(deep2) == ("0.067", "0.184")
assert rng([(m, c, "fiftieth") for m in LOG for c in C]) == ("0.038", "0.100")
assert rng([("IRT", c, "fiftieth") for c in C]) == ("0.096", "0.144")
assert rng([("BKT", c, "fiftieth") for c in C]) == ("0.030", "0.057")
assert rng([(m, c, "fiftieth") for m in ("AKT", "simpleKT") for c in ("half A", "bridge06")]) == ("0.083", "0.095")
assert rng([(m, c, "fiftieth") for m in ("DKT", "IEKT", "LPKT") for c in ("half A", "bridge06")]) == ("0.138", "0.184")
assert rng([(m, c, "history-twenty") for m in LOG for c in ("half A", "bridge06")]) == ("0.126", "0.186")
assert rng([(m, c, "history-twenty") for m in ("AKT", "simpleKT") for c in ("half A", "bridge06")]) == ("0.106", "0.114")
assert rng([("BKT", c, "history-twenty") for c in ("half A", "bridge06")]) == ("0.047", "0.059")
edm = [(m, "EDM Cup", "history-twenty") for (m, cc, g) in by if cc == C["EDM Cup"] and g == "main" and m != "running-proportion"]
assert rng(edm) == ("0.017", "0.067")
print("Every range the text states matches.")

Every range the text states matches.
